In [6]:
import json
import time
import pandas as pd
from getpass import getpass
from google import genai
from google.genai import types

MODEL_NAME = "gemini-2.5-flash"
ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

API_KEY = getpass("Paste your key: ")
client = genai.Client(api_key=API_KEY, http_options=types.HttpOptions(api_version="v1beta"))
calls = 0
print("ready")

Paste your key:  ········


ready


In [10]:
SCHEMA = {
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

SYSTEM_TEXT = "You analyse clinic call-centre notes and reply only with JSON that matches the schema."

def build_prompt(note_text):
    return f"""Context: A clinic chain in India phones patients who missed an appointment. Below is the call-centre note for one missed visit. Notes can be short, informal or in Hinglish.
Role: You are a patient-experience analyst for the clinic.
Action: Find the main reason the patient missed. Set root_cause to exactly one of: {", ".join(ALLOWED)}. If the reason is unclear or none fits, use other. Do not guess. Give confidence from 0 to 1, a summary of at most 25 words, and one practical recommended_action for the clinic.
Tone: Neutral, factual and brief.
Format: JSON with the fields root_cause, confidence, summary, recommended_action.

Note: {note_text}"""

def ask_gemini(prompt):
    global calls
    calls += 1
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SYSTEM_TEXT, temperature=0,
            max_output_tokens=1000, response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_budget=0)))
    return response.text

def analyse_note(note_text):
    return json.loads(ask_gemini(build_prompt(note_text)))

print("defined")

defined


In [11]:
notes = pd.read_csv("../data/followup_notes.csv")
n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
print(n001)

result = analyse_note(n001)
print(result)
print(list(result.keys()))
print("calls so far:", calls)

Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'Patient missed appointment due to not receiving a reminder message and forgetting.', 'recommended_action': 'Ensure reminder messages are sent reliably to patients.'}
['root_cause', 'confidence', 'summary', 'recommended_action']
calls so far: 3


In [12]:
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def is_valid(data):
    try:
        FollowupInsight(**data)
    except ValidationError:
        return False
    return data["root_cause"] in ALLOWED

print(is_valid(result))

True


In [13]:
fields = ["root_cause", "confidence", "summary", "recommended_action"]
rows = []

for _, r in notes.head(10).iterrows():
    note = r["note_text"]
    row = {"note_id": r["note_id"], "appointment_id": r["appointment_id"]}

    if pd.isna(note) or str(note).strip() == "":
        row.update(root_cause="other", confidence=0.0, summary="No note recorded",
                   recommended_action="Re-call patient", validation_status="skipped_blank", raw_response="")
    else:
        raw = ""
        try:
            raw = ask_gemini(build_prompt(note))
            data = json.loads(raw)
            for k in fields:
                row[k] = data.get(k)
            row["validation_status"] = "valid" if is_valid(data) else "invalid"
        except Exception as e:
            print(r["note_id"], "error:", e)
            row["validation_status"] = "invalid"
        row["raw_response"] = raw
        time.sleep(13)

    print(r["note_id"], row.get("root_cause"), row["validation_status"])
    rows.append(row)

out = pd.DataFrame(rows, columns=["note_id", "appointment_id"] + fields + ["validation_status", "raw_response"])
out.to_csv("../outputs/notes_analysed.csv", index=False)
print(out.shape)
print("calls so far:", calls)

N001 forgot_or_no_reminder valid
N002 distance_or_transport valid
N003 cost_or_fee valid
N004 other skipped_blank
N005 illness_or_emergency valid
N006 work_or_schedule valid
N007 forgot_or_no_reminder valid
N008 distance_or_transport valid
N009 other valid
N010 work_or_schedule valid
(10, 8)
calls so far: 12


In [14]:
bad_replies = [
    {"root_cause": "cost_or_fee", "confidence": 1.7, "summary": "Fee was too high for the patient.", "recommended_action": "Offer a payment plan."},
    {"root_cause": "cost_or_fee", "confidence": 0.8, "summary": "Fee was too high for the patient."},
]

for bad in bad_replies:
    try:
        FollowupInsight(**bad)
        print("passed")
    except ValidationError as e:
        print(e)
        print()

1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f... high for the patient.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing



In [15]:
compare = []
for nid in ["N007", "N009"]:
    note = notes.loc[notes["note_id"] == nid, "note_text"].iloc[0]
    print(nid, ":", note)

    v1 = json.loads(ask_gemini("Classify this note: " + note))
    time.sleep(13)
    v2 = out.loc[out["note_id"] == nid, "root_cause"].iloc[0]

    compare.append({"note_id": nid, "prompt": "V1 bare", "root_cause": v1.get("root_cause"), "allowed": v1.get("root_cause") in ALLOWED})
    compare.append({"note_id": nid, "prompt": "V2 CRAFT", "root_cause": v2, "allowed": v2 in ALLOWED})

print(pd.DataFrame(compare))
print("calls so far:", calls)

N007 : Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.
N009 : Phone was switched off, could not reach the patient after three attempts.
  note_id    prompt                                         root_cause  \
0    N007   V1 bare  Patient forgot appointment, did not receive SM...   
1    N007  V2 CRAFT                              forgot_or_no_reminder   
2    N009   V1 bare                                Patient unreachable   
3    N009  V2 CRAFT                                              other   

   allowed  
0    False  
1     True  
2    False  
3     True  
calls so far: 14


### Task 4.3 – Prompt comparison and call count

| note | V1 bare | allowed | V2 CRAFT | allowed |
|---|---|---|---|---|
| N007 (Hinglish) | free-text description | No | forgot_or_no_reminder | Yes |
| N009 (ambiguous) | Patient unreachable | No | other | Yes |

I would deploy V2. The bare prompt invents its own labels, which can't be counted or grouped, while the CRAFT prompt with the allowed list and the "answer other rather than guess" rule returned a usable category for both the Hinglish and the ambiguous note.

API calls: the counter shows 14 (1 in 4.1, 9 in 4.2, 2 V1 calls here; V2 reused the 4.2 results), which includes 2 setup requests the API rejected before the model ran. One more rejected request happened before the counter was reset, so 15 requests in total were sent — below the 25 limit.

In [16]:
appts = pd.read_csv("../outputs/appointments_clean.csv")
merged = out.merge(appts, on="appointment_id")
print(merged.shape[0])
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))
print((merged["reminder_sent"] == "No").sum())

10
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1
5
